# Strategy Comparison

In [23]:
import torch
from torch import nn
from torchvision import models
import torch_mlir
import numpy as np
import iree.compiler
import iree.runtime

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def compile_str(mlir):
    return iree.runtime.load_vm_flatbuffer(
        iree.compiler.compile_str(
            mlir, input_type="tosa", target_backends=["llvm-cpu"],
            extra_args=[
                "--iree-llvmcpu-target-cpu-features=host",
                "--iree-stream-partitioning-favor=max-concurrency",
                "--iree-flow-zero-fill-empty-tensors",
                "--iree-llvmcpu-fail-on-out-of-bounds-stack-allocation=0",
                "--iree-opt-const-eval",
                "--iree-opt-const-expr-hoisting",
                "--iree-opt-numeric-precision-reduction",
                "--iree-opt-strip-assertions"
            ]
        ),
        backend="llvm-cpu"
    )

def compile_file(filename):
    with open(filename) as f:
        return compile_str(f.read())
    
def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

# def timeit(stmt, n=100):
#     return ti(stmt, globals=globals(), number=n) * 1000 / n

def cal_memory_usage(lines):
    usage = 0
    for line in lines:
        line = line[line.find("tensor") + 7:]
        line = line[:line.find("f32") - 1]
        line = line.replace("x", " ").strip().split(" ")
        for x in line:
            usage += int(x) * 4
    return usage

def get_model_df(model):
    df = pd.DataFrame()
    image = torch.randn(1, 3, 224, 224).numpy()
    grad = torch.randn(1, 1000).numpy()

    strategies = ["recompute", "checkpoint", "storeall", "heuristic"]

    mlirs = []

    for strategy in strategies:
        with open(f"{model}/{strategy}.mlir") as f:
            mlirs.append(f.read())

    mlirs = [list(filter(lambda x: "ml_program.global " in x, mlir.splitlines())) for mlir in mlirs]
    usage = [cal_memory_usage(mlir) for mlir in mlirs]

    recompute, checkpoint, storeall, heuristic = [
        compile_file(f"{model}/{x}.mlir") for x in strategies
    ]

    for strategy, mem in zip(strategies, usage):
        f = timeit(f"{strategy}.forward(image)", globals=locals(), number=1)
        b = timeit(f"{strategy}.dforward(grad)", globals=locals(), number=100) / 100
        new_df = get_dataframe(0, b, strategy.title())
        new_df["mem"] = mem
        df = pd.concat([df, new_df])

    return df[df["pass"] == "Backward"]

## ResNet

In [26]:
resnet18_df = get_model_df("resnet18")
resnet34_df = get_model_df("resnet34")
resnet50_df = get_model_df("resnet50")
resnet101_df = get_model_df("resnet101")
resnet152_df = get_model_df("resnet152")

## ViT (TODO)

## Visualization

In [41]:
fig, axs = plt.subplots(3, 2)
axs[2, 1] = None
((ax0, ax1), (ax2, ax3), (ax4, ax5)) = axs
subplots = [ax0, ax1, ax2, ax3, ax4, ax5]

def plot(df, ax):
    bar = sns.barplot(df, x="item", y="time", ax=ax)
    bar.set_xlabel("Strategy")
    bar.set_ylabel("Time Cost")
    
    line = bar.twinx()
    sns.lineplot(df, x="item", y="mem", ax=line, color="#FFAA55", linewidth=3)
    line.set_ylabel("Time Cost")
    ax.set_title("Taping Comparison")

plt.rcParams["figure.dpi"] = 300
plt.rcParams["figure.figsize"] = (30, 45)
plt.style.use("seaborn-v0_8-pastel")

# bar = sns.barplot(df, x="item", y="time")
# plt.xlabel("Strategy")
# plt.ylabel("Time Cost")

# line = bar.twinx()
# sns.lineplot(df, x="item", y="mem", ax=line, color="#FFAA55", linewidth=3)
# plt.ylabel("Tape Overhead")
# plt.title("Strategy Comparison")

# plt.savefig("strategy-comparison.png", bbox_inches="tight", dpi=300, pad_inches=0.1)

for df, ax in zip([resnet18_df, resnet34_df, resnet50_df, resnet101_df, resnet152_df], subplots):
    plot(df, ax)
    
plt.savefig("taping.png", bbox_inches="tight", dpi=300, pad_inches=0.1)

In [36]:
resnet18_df

,time,pass,item,mem
0,0.041092,Backward,Recompute,1808
0,0.036245,Backward,Checkpoint,20180
0,0.022109,Backward,Storeall,74520
0,0.022211,Backward,Heuristic,45632


In [37]:
resnet34_df

,time,pass,item,mem
0,0.060426,Backward,Recompute,1808
0,0.051445,Backward,Checkpoint,21580
0,0.022421,Backward,Storeall,109944
0,0.020985,Backward,Heuristic,81056


In [38]:
resnet50_df

,time,pass,item,mem
0,0.138179,Backward,Recompute,1808
0,0.130508,Backward,Checkpoint,56772
0,0.104999,Backward,Storeall,329056
0,0.100491,Backward,Heuristic,221272


In [39]:
resnet101_df

,time,pass,item,mem
0,0.216305,Backward,Recompute,1808
0,0.199146,Backward,Checkpoint,61080
0,0.119982,Backward,Storeall,549784
0,0.119268,Backward,Heuristic,442000


In [40]:
resnet152_df

,time,pass,item,mem
0,0.313784,Backward,Recompute,1808
0,0.277012,Backward,Checkpoint,85756
0,0.134581,Backward,Storeall,748624
0,0.134505,Backward,Heuristic,640840
